# Vehicle routing

Run the full original two-vehicle, four-city assignment and inspect actual finite-shot observations.

## Preserve the logical city and bit mapping

The visible qfunc prepares sixteen decisions and twelve cost/mixer layers
from freshly generated departure, arrival and intercity distances. FlagQuantum
constructs the coherent cost and named output. Host problem.py supplies the
two-vehicle route rules and weights; quantum.py explicitly converts native
array order to the cost table's logical word order.

Try a nonzero twenty-four-angle preview binding before changing city data.
Propagate program edits to shared main.py to affect the full optimizer.
The retained 26 objective queries and 30000 final draws are observations, not
proof of a best feasible route. Verify both vehicle routes and constraint
residuals before interpreting a low sampled scalar cost.


Load the native problem, circuit compiler, and workflow.

In [ ]:
from flagquantum import synth
from flagquantum.synth import Output
from flagquantum.synth import QArray
from flagquantum.synth import QBit
from flagquantum.synth import RX
from flagquantum.synth import allocate
from flagquantum.synth import hadamard_transform
from flagquantum.synth import qfunc

from examples.synthesis.applications.logistics.vehicle_routing_problem.main import cost
from examples.synthesis.applications.logistics.vehicle_routing_problem.overview.quantum import (
    LAYERS,
    Problem,
)


@qfunc
def native_main(
    params: list[float],
    departure: list[float],
    arrival: list[float],
    distances: list[float],
    x: Output[QArray[QBit]],
) -> None:
    allocate(16, x)
    hadamard_transform(x)
    for layer in range(12):
        cost(params[2 * layer], departure, arrival, distances, x)
        for bit in range(16):
            RX(0.1 * params[2 * layer + 1], x[bit])


def build_preview(problem: Problem, parameters: tuple[float, ...]) -> synth.Program:
    if len(parameters) != 2 * LAYERS:
        raise ValueError("twelve layers require twenty-four parameters")
    return synth.synthesize(
        native_main,
        classical_inputs={
            "params": list(parameters),
            "departure": [problem.distances[0][city] for city in range(2, 6)],
            "arrival": [problem.distances[1][city] for city in range(2, 6)],
            "distances": [
                problem.distances[first][second]
                for first in range(2, 6)
                for second in range(2, 6)
            ],
        },
        max_qubits=16,
        max_workspace=0,
        memory_limit_bytes=64 * 1024**2,
    )


from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.logistics.vehicle_routing_problem.overview import (
    main,
    plots,
    problem,
    quantum,
    workflow,
)


Keep two cities per vehicle.

In [ ]:
cities_per_vehicle = 2


Keep four positions, including each depot at both ends.

In [ ]:
positions = tuple(range(cities_per_vehicle + 2))


Retain every route position.

In [ ]:
assert positions == (0, 1, 2, 3)


Inspect all four route positions.

In [ ]:
print(positions)


Choose both interior positions.

In [ ]:
inner_positions = positions[1:-1]
print(inner_positions)


Generate the original six positions from RandomState(10).

In [ ]:
data = problem.fresh()


Retain every original coordinate for the source graph.

In [ ]:
assert len(data.positions) == 6


Compute all original Euclidean distances and their normalization.

In [ ]:
distances = data.distances


Inspect the unscaled six-location geometry.

In [ ]:
print(data.positions)


Use the original six-location distance normalization.

In [ ]:
print(data.scale)


Retain normalized distances for every pair.

In [ ]:
assert len(distances) == 6 and all(len(row) == 6 for row in distances)


Inspect normalized distances.

In [ ]:
print(distances)


Retain the original global scale.

In [ ]:
global_scale = 0.1


Retain the relative objective scale.

In [ ]:
relative_objective = 1.5


Form the objective normalization.

In [ ]:
assert abs(global_scale * relative_objective - 0.15) < 1e-15


Retain the source constraint ratio.

In [ ]:
relative_constraint = 6


Form the source penalty scale.

In [ ]:
assert abs(global_scale * relative_objective * relative_constraint - 0.9) < 1e-15


Use the original city/position/vehicle indicator.

In [ ]:
indicator = problem.indicator


Validate the two classes of assignment constraints.

In [ ]:
check_validity = problem.valid


Keep the original travel objective in the scalar oracle.

In [ ]:
scalar_cost = problem.scalar_cost


Evaluate every assignment's travel and penalty costs.

In [ ]:
all_costs = problem.costs(data)


Check the complete original sixteen-bit decision domain.

In [ ]:
assert all_costs.numel() == 65536


Use the native twelve-layer qfunc.

In [ ]:
assert problem.LAYERS == 12


Synthesize the complete native circuit with fresh distances.

In [ ]:
initial = quantum.initial_parameters()
program = build_preview(data, initial)
print(program.report["resources"])


Keep the source sample size and initialization schedule.

In [ ]:
assert problem.SHOTS == 30000 and len(initial) == 24


Use original costs for measured outcomes.

In [ ]:
cost_function = problem.scalar_cost


Retain actual optimization observations.

In [ ]:
run = workflow.optimize


Run the original observed 26-query finite-shot budget.

In [ ]:
result = run(data)


Retain the recorded sampled objective trace.

In [ ]:
assert len(result.trace) == 26


Plot actual optimized angles alongside the source locations.

In [ ]:
figure_directory = Path("routing-figures")
plots.save(result, figure_directory)
display(Image(filename=str(figure_directory / "locations.png")))
display(Image(filename=str(figure_directory / "parameters.png")))


Read the actual final 30,000-shot distribution.

In [ ]:
assert sum(count for _, count in result.counts) == 30000


Sort observed outcomes by the original objective.

In [ ]:
main.report(result)


Select the lowest-cost actually observed assignment.

In [ ]:
selected = result.selected_word


Decode each vehicle's route.

In [ ]:
routes = tuple(problem.route(selected, vehicle) for vehicle in range(2))
assert len(routes) == 2


Plot the original geometry and actual observed routes.

In [ ]:
display(Image(filename=str(figure_directory / "routes.png")))
